In [31]:
# 1-1
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_1.csv')
# print(df.head())
# print(df.info())

df1 = df[(df['ProductA가격'] != 0) & (df['ProductB가격'] != 0)]
# print(df1.info())
df1['차이'] = np.abs(df1['ProductA가격'] - df1['ProductB가격'])
print(np.round(df1.groupby('도시명').mean()['차이'].max(), 1))

16333.3


C:\Users\user\AppData\Local\Temp\ipykernel_12708\2129511932.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['차이'] = np.abs(df1['ProductA가격'] - df1['ProductB가격'])


In [23]:
# 1-2
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_2.csv')
# print(df.head())
# print(df.info())

df['Height_m'] = df['Height_cm'] / 100
df['BMI'] = df['Weight_kg'] / (df['Height_m'] ** 2)
print(df[(df['BMI'] < 18.5) | (df['BMI'] >= 25)].count())

Height_cm    74
Weight_kg    74
Height_m     74
BMI          74
dtype: int64


In [96]:
# 1-3
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_3.csv')

# print(df.head())
# print(df.info())
df['순생산량'] = (df['products_made_domestic'] + df['products_made_international']) - (df['products_sold_domestic'] + df['products_sold_international'])

# 1. 맹점 방지: 만약 같은 연도/같은 공장의 데이터가 여러 줄일 수 있으므로 먼저 합쳐줍니다.
factory_yearly_sum = df.groupby(['year', 'factory'])['순생산량'].sum().reset_index()

# 2. 연도(year)별로 가장 높은 순생산량을 기록한 행의 인덱스를 찾습니다.
max_idx = factory_yearly_sum.groupby('year')['순생산량'].idxmax()

# 3. 찾아낸 인덱스를 바탕으로 연도별 1등 공장의 데이터만 필터링합니다.
best_factories = factory_yearly_sum.loc[max_idx]

# 4. 필터링된 1등 공장들의 순생산량을 모두 더합니다.
total_sum = best_factories['순생산량'].sum()

print(total_sum)


9130


In [ ]:
# 1-3
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_1_3.csv')
# print(df.head())
# print(df.info())



    year    factory  순생산량
0   2020  Factory A  1032
1   2020  Factory B   328
2   2020  Factory C  -212
3   2020  Factory D   784
4   2020  Factory E  2034
5   2021  Factory A  1937
6   2021  Factory B  -677
7   2021  Factory C  -915
8   2021  Factory D  -469
9   2021  Factory E  1222
10  2022  Factory A  1288
11  2022  Factory B  -523
12  2022  Factory C   688
13  2022  Factory D  2166
14  2022  Factory E   -24
15  2023  Factory A  2691
16  2023  Factory B -1921
17  2023  Factory C  2052
18  2023  Factory D  2517
19  2023  Factory E  2993
year
2020     4
2021     5
2022    13
2023    19
Name: 순생산량, dtype: int64


np.int64(9130)

In [ ]:
# 2
import numpy as np
import pandas as pd

df_train = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_train.csv')
df_test = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_test.csv')

# print(df_train.info())
# print(df_train.head())

train_y = df_train['DBP']
train_X = df_train.drop(columns=['DBP', 'level_0', 'ID'])
test_y = df_test['DBP']
test_X = df_test.drop(columns=['DBP', 'level_0', 'ID'])

from sklearn.model_selection import train_test_split

train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, valid_X.shape, train_y.shape, valid_y.shape)

# train_X.info()
train_X_cat = train_X.select_dtypes('object').columns.tolist()
train_X_num = train_X.select_dtypes('number').columns.tolist()
valid_X_cat = valid_X.select_dtypes('object').columns.tolist()
valid_X_num = valid_X.select_dtypes('number').columns.tolist()
test_X_cat = test_X.select_dtypes('object').columns.tolist()
test_X_num = test_X.select_dtypes('number').columns.tolist()

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

imputer = SimpleImputer(strategy='most_frequent')
imputer_train_X = imputer.fit_transform(train_X[train_X_cat])
imputer_valid_X = imputer.transform(valid_X[valid_X_cat])
imputer_test_X = imputer.transform(test_X[test_X_cat])

onehotencoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
one_train_X = onehotencoder.fit_transform(imputer_train_X)
one_valid_X = onehotencoder.transform(imputer_valid_X)
one_test_X = onehotencoder.transform(imputer_test_X)

train_X = np.concatenate([one_train_X, train_X[train_X_num]], axis=1)
valid_X = np.concatenate([one_valid_X, valid_X[valid_X_num]], axis=1)
test_X = np.concatenate([one_test_X, test_X[test_X_num]], axis=1)

from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor()
rf.fit(train_X, train_y)

pred = rf.predict(valid_X)

from sklearn.metrics import root_mean_squared_error
# print(root_mean_squared_error(valid_y, pred))
# print(df_train['DBP'].std())

pred2 = rf.predict(test_X)
print(root_mean_squared_error(test_y, pred2))

pred2 = pd.DataFrame(pred2, columns=['pred'])
pred2.to_csv('result.csv', index=False)

10.689879296785579


In [ ]:
# 3-1-1
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_1.csv')
# print(df.head())
# print(df.info())

np.round(df['work_hours'].mean(), 3)

np.float64(8.09)

In [30]:
# 3-1-2
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_1.csv')
# print(df.head())
# print(df.info())

np.round(df['work_hours'].std(), 3)

np.float64(1.519)

In [11]:
# 3-1-3
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_1.csv')

mu = np.round(df['work_hours'].mean(), 3)
sigma = np.round(df['work_hours'].std(), 3)

from scipy.stats import kstest, norm
statistic, p_value = kstest(df['work_hours'], 'norm', args=(mu, sigma))
print(np.round(statistic, 3))

# -------------------------------------------------------------
'''
K-S 검정(Kolmogorov-Smirnov test)은 “데이터의 분포가 어떤 이론분포와 같은지”를 보는 비모수 검정이다.

원래는 그냥 평균 0 표준편차1로 해서 ks검정을 하는데 지금은 단위가 있는 형태라서 
따로 그 단위에 맞는 평균값과 표준편차값을 넣어서 진행함.

'''

0.064


'\nK-S 검정(Kolmogorov-Smirnov test)은 “데이터의 분포가 어떤 이론분포와 같은지”를 보는 비모수 검정이다.\n\n원래는 그냥 평균 0 표준편차1로 해서 ks검정을 하는데 지금은 단위가 있는 형태라서 \n따로 그 단위에 맞는 평균값과 표준편차값을 넣어서 진행함.\n\n'

In [1]:
# 3-1-4
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_1.csv')

mu = np.round(df['work_hours'].mean(), 3)
sigma = np.round(df['work_hours'].std(), 3)

from scipy.stats import kstest, norm
statistic, p_value = kstest(df['work_hours'], 'norm', args=(mu, sigma))
print(p_value)

0.777349235942209


In [12]:
# 3-2-1
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_2.csv')
# print(df.head())
# print(df.info())

import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.ols("price ~ area + rooms + age", data=df).fit()
# print(model.summary())
print(model.params[1:].idxmax())

rooms


In [ ]:
# 3-2-2
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_3_2.csv')
# print(df.head())
# print(df.info())

import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.ols("price ~ area + rooms + age", data=df).fit()
# print(model.summary())
pvalues = model.pvalues[1:]
print(pvalues[pvalues < 0.05].sum())

3
